# Adicionando funções externas a API da OpenAI

Um grande salto de possibilidades de utlizações únicas da LLMs ocorreu quando a OpenAI lançou o `function calling`. Essa ferramenta permite adicionarmos manualmente funções externas ao modelo que ele, dependendo da situação, poderá utilizar para obter novas informações ou atuar em diversos escopos.
Vamos fazer uma breve revisão de como utilizamos funções externas na api da OpenAI, este assunto é explocado mais afundo no curso de Explorando a API da OpenAI. Na próxima aula, mostraremos como o framework LangChain facilita a utilização das funções externas.

## Importações iniciais

In [51]:
import json

import openai
from dotenv import load_dotenv

_ = load_dotenv()  # carrega variáveis de ambiente do arquivo .env

client = openai.Client()  # cria o cliente da OpenAI

## Criando função que será adicionada ao modelo

Utilizaremos uma função simples que simula uma API de tempo, que retorna a temperatura de um determinado local. Lembrando que, modelos de LLM são treinados com dados históricos, portanto, não possuem informações atuais. A única forma de eles entenderem o que está ocorrendo neste instante é passando informações para eles através de prompts ou de funções externas.

In [52]:
def obter_temperatura_atual(local, unidade="celsius"):
    if "são paulo" in local.lower():
        return json.dumps({
            "local": "São Paulo",
            "temperatura": "32",
            "unidade": unidade
        })
    elif "porto alegre" in local.lower():
        return json.dumps({
            "local": "Porto Alegre",
            "temperatura": "25",
            "unidade": unidade
        })
    else:
        return json.dumps({
            "local": local,
            "temperatura": "unknown",
        })

In [53]:
obter_temperatura_atual("Porto Alegre")

'{"local": "Porto Alegre", "temperatura": "25", "unidade": "celsius"}'

## Criando descrição da função

Através dessa descrição o modelo entenderá o que a função faz e como ela pode ser utilizada.

In [54]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "obter_temperatura_atual",
            "description": "Obtém a temperatura atual em uma dada cidade",
            "parameters": {
                "type": "object",
                "properties": {
                    "local": {
                        "type": "string",
                        "description": "O nome da cidade. Ex: São Paulo",
                    },
                    "unidade": {
                        "type": "string", 
                        "enum": ["celsius", "fahrenheit"]
                    },
                },
                "required": ["local"],
            },
        },
    }
    ]

## Chamando o modelo com a nova ferramenta

Para chamar o modelo com a ferramenta criada, basta passar o argumento tools com uma lista de ferramentas.

In [55]:
mensagens = [
  {"role": "user", "content": "Qual a temperatura atual em Porto Alegre?"}
]

resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
    messages=mensagens,
    tools=tools,
    tool_choice="auto"
)

resposta

ChatCompletion(id='chatcmpl-CcfUogmUMfnOk3BdoOcB0uSnixRd6', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FFHSvO04a2G6shIx05pZELcv', function=Function(arguments='{"local":"Porto Alegre","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')]))], created=1763332010, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=28, prompt_tokens=87, total_tokens=115, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [56]:
mensagem = resposta.choices[0].message
mensagem

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FFHSvO04a2G6shIx05pZELcv', function=Function(arguments='{"local":"Porto Alegre","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')])

In [57]:
mensagem.content

In [58]:
mensagem.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_FFHSvO04a2G6shIx05pZELcv', function=Function(arguments='{"local":"Porto Alegre","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')]

In [59]:
tools_call = mensagem.tool_calls[0]
print(tools_call.function.name)
print(tools_call.function.arguments)

obter_temperatura_atual
{"local":"Porto Alegre","unidade":"celsius"}


### Adicionando resultado da função as mensagens

O operador ** em Python é usado para desempacotar (unpack) um dicionário em argumentos nomeados de uma função. No contexto do seu código:

```py
def foo(a, b):
    print(a, b)

args = {"a": 1, "b": 2}
foo(**args)  # Equivalente a foo(a=1, b=2)
```

In [60]:
observacao = obter_temperatura_atual(**json.loads(tools_call.function.arguments))
observacao

'{"local": "Porto Alegre", "temperatura": "25", "unidade": "celsius"}'

### Chamando novamente o modelo

In [61]:
mensagens.append(mensagem)
mensagens

[{'role': 'user', 'content': 'Qual a temperatura atual em Porto Alegre?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FFHSvO04a2G6shIx05pZELcv', function=Function(arguments='{"local":"Porto Alegre","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')])]

In [ ]:
mensagens.append({
  "tool_call_id": tools_call.id,
  "role": "tool",
  "name": tools_call.function.name,
  "content": observacao
})
mensagens

[{'role': 'user', 'content': 'Qual a temperatura atual em Porto Alegre?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FFHSvO04a2G6shIx05pZELcv', function=Function(arguments='{"local":"Porto Alegre","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')]),
 {'tool_call_id': 'call_FFHSvO04a2G6shIx05pZELcv',
  'role': 'tool',
  'name': 'obter_temperatura_atual',
  'content': '{"local": "Porto Alegre", "temperatura": "25", "unidade": "celsius"}'}]

In [66]:
resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
	messages=mensagens,
	tools=tools,
	tool_choice="auto"
)
resposta

ChatCompletion(id='chatcmpl-CcfmR9yFTfqmYJt9svPIwhEX4IPkW', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A temperatura atual em Porto Alegre é de 25°C.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1763333103, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=15, prompt_tokens=153, total_tokens=168, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [68]:
resposta.choices[0].message.content

'A temperatura atual em Porto Alegre é de 25°C.'

# Explorando diferentes perguntas e o parâmetro tool_choice

Através do parâmetro tool_choice é possível forçar o modelo a sempre utilizar uma tool. Vamos ver como ele se comporta para diferentes perguntas modificando o parâmetro.

## Parâmetro "auto"

Assim o modelo define automaticamente se é necessário a utilização de uma função ou não.

In [69]:
mensagens = [
  {"role": "user", "content": "Qual a temperatura em Porto Alegre?"}
]
resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
	messages=mensagens,
	tools=tools,
	tool_choice="auto"
)
mensagem = resposta.choices[0].message
print("Conteúdo:", mensagem.content)
print("Tools:", mensagem.tool_calls)

Conteúdo: None
Tools: [ChatCompletionMessageFunctionToolCall(id='call_Wns03SQPmEty12fKodbA9wss', function=Function(arguments='{"local":"Porto Alegre","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')]


In [70]:
mensagens = [
  {"role": "user", "content": "Olá"}
]
resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
	messages=mensagens,
	tools=tools,
	tool_choice="auto"
)
mensagem = resposta.choices[0].message
print("Conteúdo:", mensagem.content)
print("Tools:", mensagem.tool_calls)

Conteúdo: Olá! Como posso te ajudar hoje?
Tools: None


## Parâmetros "None"

Com o parâmetro "None", o modelo não vai utilizar funções.

In [72]:
mensagens = [
  {"role": "user", "content": "Qual a temperatura em Porto Alegre?"}
]
resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
	messages=mensagens,
	tools=tools,
	tool_choice="none"
)
mensagem = resposta.choices[0].message
print("Conteúdo:", mensagem.content)
print("Tools:", mensagem.tool_calls)

Conteúdo: Vou verificar a temperatura atual em Porto Alegre para você. Apenas um momento, por favor.
Tools: None


In [73]:
mensagens = [
  {"role": "user", "content": "Olá"}
]
resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
	messages=mensagens,
	tools=tools,
	tool_choice="none"
)
mensagem = resposta.choices[0].message
print("Conteúdo:", mensagem.content)
print("Tools:", mensagem.tool_calls)

Conteúdo: Olá! Como posso ajudar você hoje?
Tools: None


## Parâmetro "function"

Podemos fazer o modelo rodar obrigatoriamente a função, passando dentro de um dicionário a função que o modelo deve rodar.

In [77]:
mensagens = [
  {"role": "user", "content": "Qual a temperatura em Porto Alegre?"}
]
resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
	messages=mensagens,
	tools=tools,
	tool_choice={'type': 'function', 'function':{'name': 'obter_temperatura_atual'}}
)
mensagem = resposta.choices[0].message
print("Conteúdo:", mensagem.content)
print("Tools:", mensagem.tool_calls)

Conteúdo: None
Tools: [ChatCompletionMessageFunctionToolCall(id='call_0nJEDQt3sFIh9qTX2nCoU2qf', function=Function(arguments='{"local":"Porto Alegre"}', name='obter_temperatura_atual'), type='function')]


In [78]:
mensagens = [
  {"role": "user", "content": "Olá"}
]
resposta = client.chat.completions.create(
	model="gpt-3.5-turbo-0125",
	messages=mensagens,
	tools=tools,
	tool_choice={'type': 'function', 'function':{'name': 'obter_temperatura_atual'}}
)
mensagem = resposta.choices[0].message
print("Conteúdo:", mensagem.content)
print("Tools:", mensagem.tool_calls)

Conteúdo: None
Tools: [ChatCompletionMessageFunctionToolCall(id='call_NbddACcpRvEO4wwuG3oxWTmV', function=Function(arguments='{"local":"São Paulo","unidade":"celsius"}', name='obter_temperatura_atual'), type='function')]
